# Exercise 3 – Task: does the light change the picture?

> **Try it yourself in Google Flow:** open [Google Flow](https://labs.google/flow), paste the prompts from this notebook and create your own pictures. Images in Google Flow are free. Do not create videos yet.

**Goal:** your own experiment. Change the **light** block: version A at dawn, version B at night. Create **two pictures per version** and rate them – by eye and with the OpenAI API: *does the picture show night?* Then compare the share of night pictures per version.

Build it yourself in three steps. Every step can be done by **copying code from `image_generation_example.ipynb` and adapting it**.

Each step: write your code in the empty cell → run the **check** (✓ go on, ✗ shows what is missing; a `NameError` means the variable does not exist yet). Stuck or out of time? Open the **hint**, then the **solution**.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (models `gpt-image-2` to create pictures and `gpt-6-luna` to rate them); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys

import pandas as pd
from pydantic import BaseModel

# helper.py and checks.py are in the main folder, one level up.
sys.path.append("..")
from helper import ask, create_pictures, show_images
from checks import check_pictures, check_prompts, check_ratings

## Step 1: two prompts

Build `prompt_a` with the light block `"at dawn, soft light"` and `prompt_b` with `"at night, in moonlight"`. All other blocks stay the same.

In [ ]:
# Your code here.


In [ ]:
check_prompts(prompt_a, prompt_b, subject, setting, composition)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 1 – change the block `light` instead of `setting`.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
subject = "a small red wooden rowboat with two oars"
setting = "on a mountain lake"
light = "at dawn, soft light"
composition = "wide shot"
prompt_a = f"{subject}, {setting}, {light}, {composition}"

light = "at night, in moonlight"  # change one block only
prompt_b = f"{subject}, {setting}, {light}, {composition}"

print("A:", prompt_a)
print("B:", prompt_b)
```

</details>

## Step 2: two pictures per version

Create four pictures: `variant_a_1` and `variant_a_2` from prompt A, `variant_b_1` and `variant_b_2` from prompt B. Store the names in `names` and the file paths in `paths`, then show them – version A in the top row. All four cost about US$0.025.

In [ ]:
# Your code here.


In [ ]:
check_pictures(paths)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 2: `create_pictures()` takes a list of names and a list of prompts of the same length – use each prompt twice.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
names = ["variant_a_1", "variant_a_2", "variant_b_1", "variant_b_2"]
prompts = [prompt_a, prompt_a, prompt_b, prompt_b]

paths = create_pictures(names, prompts, "variants")
show_images(paths, columns=2)
```

</details>

## Step 3: rate and compare

Build the DataFrame `ratings` with one row per picture and the columns:

- `name` and `version` (`"A"` or `"B"`),
- `me`: **your** rating by eye – look at the pictures: night, `True` or `False`?
- `api`: the rating of the OpenAI API for the same question.

Then print the share of night pictures per version: `ratings.groupby("version")[["me", "api"]].mean()`.

In [ ]:
# Your code here.


In [ ]:
check_ratings(ratings)

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hint</summary>

Example, section 3: the same `Rating` class and loop – collect `answer["matches"]` in a list instead of printing it. Build the table like in Exercise 1, one column at a time.

</details>

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Solution</summary>

```python
class Rating(BaseModel):
    matches: bool
    reason: str


PROMPT = "Does the image show night, with a dark sky or moonlight? Give a short reason."

ratings = pd.DataFrame()
ratings["name"] = names
ratings["version"] = ["A", "A", "B", "B"]
ratings["me"] = [False, False, True, True]  # your own rating, by eye

api = []
for path in paths:
    api.append(ask(PROMPT, Rating, [path])["matches"])
ratings["api"] = api

print(ratings.groupby("version")[["me", "api"]].mean())
ratings
```

</details>

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why does the same prompt give different pictures – and what does that mean for your experiment?
2. Why do we change only one building block between version A and version B?
3. With two pictures per version, the share can only be 0, 1/2 or 1. What does this mean for your conclusion?
4. Where do your rating and the OpenAI rating differ? Who is right – and why can you not simply trust the API rating?
5. Google Flow and the OpenAI image API have no seed setting. What do you write down so that someone else can follow your experiment?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Generation starts from random noise, so every run is different. One picture per version proves nothing. You need several pictures per version and compare the shares.

2. If several blocks change, you cannot tell which one caused the difference. One change per round makes it an experiment instead of trial and error.

3. Two pictures are weak evidence: a single picture moves the share by half. Describe the result as a tendency. A clear conclusion needs more pictures.

4. Look at the pictures where the two ratings differ. Borderline cases – a few waves, a cloudy sky – are hard to judge, for you and for the model. Your own judgement is the reference, so always check a sample by hand.

5. The prompt, the model, the settings (e.g. aspect ratio) and the date. Keep all pictures – also the ones that did not work.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')